**Setup.Install and import libraries**

In [ ]:
# Install RDKit in Colab
!pip -q install rdkit

# Basic libraries
import os

import numpy as np
import pandas as pd

# RDKit libraries
from rdkit import Chem
from rdkit import DataStructs
from rdkit import RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# Reduce unnecessary RDKit warning messages
RDLogger.DisableLog("rdApp.warning")

# Reproducibility
RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


**Step 1. Load the QSAR dataset**

In [ ]:
df = pd.read_csv("DPP4_QSAR_dataset_parent_cleaned_locked.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (3666, 25)


,parent_smiles,molecule_chembl_id,pref_name,canonical_smiles_examples,n_records,IC50_nM_median,IC50_nM_min,IC50_nM_max,pIC50_median,pIC50_min,pIC50_max,pIC50_std,pchembl_median,assay_type,assay_chembl_id,document_chembl_id,mw_freebase,alogp,hba,hbd,psa,rtb,num_ro5_violations,pIC50_range,activity_class
0,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,CHEMBL4246951,NaN,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,1,28050.00,28050.00,28050.00,4.552067,4.552067,4.552067,0.0,4.55,B,CHEMBL4232372,CHEMBL4229387,318.13,2.62,6.0,1.0,68.24,1.0,0.0,0.0,Low
1,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H](C)N1,CHEMBL260701,NaN,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H](C)N1,1,60.00,60.00,60.00,7.221849,7.221849,7.221849,0.0,7.22,B,CHEMBL939125,CHEMBL1143890,276.34,-0.57,4.0,1.0,76.44,4.0,0.0,0.0,High
2,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1ccccc1C#N,CHEMBL3913618,NaN,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1ccccc1C#N,1,16.00,16.00,16.00,7.795880,7.795880,7.795880,0.0,7.80,B,CHEMBL3888356,CHEMBL3886676,417.47,0.38,9.0,1.0,114.87,4.0,0.0,0.0,High
3,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3ccccc3n1)c(=O)n2C,CHEMBL5277490,NaN,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3ccccc3n1)c(=O)n2C,1,1.33,1.33,1.33,8.876148,8.876148,8.876148,0.0,8.88,B,CHEMBL5247172,CHEMBL5244277,458.53,0.76,10.0,1.0,116.86,4.0,0.0,0.0,High
4,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,CHEMBL1650422,NaN,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,1,5.00,5.00,5.00,8.301030,8.301030,8.301030,0.0,8.30,B,CHEMBL1664314,CHEMBL1649419,333.40,1.07,6.0,1.0,87.94,3.0,0.0,0.0,High


**Step 2. Standardize pIC50 column name**

In [ ]:
# Define expected column names
smiles_col = "parent_smiles"
activity_col = "pIC50_median"
class_col = "activity_class"

# Check required columns
required_cols = [smiles_col, activity_col]
missing_cols = [col for col in required_cols if col not in df.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Create activity class if it does not already exist
if class_col not in df.columns:
    df[class_col] = pd.cut(
        df[activity_col],
        bins=[-np.inf, 6, 7, np.inf],
        labels=["Low", "Moderate", "High"],
        right=False
    )

print("Using SMILES column:", smiles_col)
print("Using activity column:", activity_col)
print("Using class column:", class_col)

print("\nActivity summary:")
display(df[activity_col].describe())

print("\nActivity class distribution:")
display(df[class_col].value_counts())

Using SMILES column: parent_smiles
Using activity column: pIC50_median
Using class column: activity_class

Activity summary:


,pIC50_median
count,3666.000000
mean,7.111067
std,1.213927
min,4.000000
25%,6.367038
50%,7.193820
75%,8.000000
max,10.096910



Activity class distribution:


,count
activity_class,
High,2113
Moderate,908
Low,645


**Step 3. Check known DPP4 inhibitors in the dataset**

In [ ]:
known_dpp4_names = [
    "SITAGLIPTIN",
    "VILDAGLIPTIN",
    "SAXAGLIPTIN",
    "LINAGLIPTIN",
    "ALOGLIPTIN",
    "GOSOGLIPTIN",
    "TENELIGLIPTIN",
    "ANAGLIPTIN",
    "GEMIGLIPTIN",
    "EVOGLIPTIN"
]

if "pref_name" not in df.columns:
    print("Column 'pref_name' was not found in the dataset.")
    print("Known inhibitor search by name will be skipped.")
    known_hits = pd.DataFrame()

else:
    df["pref_name_upper"] = df["pref_name"].astype(str).str.upper()

    known_hits = df[
        df["pref_name_upper"].isin(known_dpp4_names)
    ].copy()

    if not known_hits.empty:
        known_hits = known_hits.sort_values(
            by=activity_col,
            ascending=False
        )

print("Known DPP-4 inhibitors found:", known_hits.shape[0])

display_cols = [
    "molecule_chembl_id",
    "pref_name",
    smiles_col,
    activity_col,
    class_col
]

display_cols = [
    col for col in display_cols
    if col in known_hits.columns
]

if not known_hits.empty:
    display(known_hits[display_cols])
else:
    print("No known DPP-4 inhibitors were found by pref_name.")

Known DPP-4 inhibitors found: 2


,molecule_chembl_id,pref_name,parent_smiles,pIC50_median,activity_class
372,CHEMBL1779710,EVOGLIPTIN,CC(C)(C)OC[C@@H]1C(=O)NCCN1C(=O)C[C@H](N)Cc1cc(F)c(F)cc1F,9.045757,High
3600,CHEMBL515387,GOSOGLIPTIN,O=C([C@@H]1C[C@H](N2CCN(c3ncccn3)CC2)CN1)N1CCC(F)(F)C1,7.918166,High


**Step 4.  Prepare RDKit molecules and calculate simple molecular descriptors**

In [ ]:
def smiles_to_mol(smiles):
    """
    Convert a SMILES string to an RDKit molecule.
    Returns None if the SMILES is missing or invalid.
    """
    if pd.isna(smiles):
        return None

    try:
        return Chem.MolFromSmiles(str(smiles))
    except:
        return None


# Convert parent SMILES to RDKit molecules
df["mol"] = df[smiles_col].apply(smiles_to_mol)

# Count invalid molecules before removing them
invalid_count = df["mol"].isna().sum()

# Remove invalid molecules
df = df.dropna(subset=["mol"]).copy()

print("Invalid molecules removed:", invalid_count)
print("Valid molecules:", df.shape[0])

# Calculate simple molecular descriptors
df["MW"] = df["mol"].apply(Descriptors.MolWt)
df["LogP"] = df["mol"].apply(Descriptors.MolLogP)
df["HBA"] = df["mol"].apply(Lipinski.NumHAcceptors)
df["HBD"] = df["mol"].apply(Lipinski.NumHDonors)

print("\nDescriptor summary:")
display(df[["MW", "LogP", "HBA", "HBD"]].describe())

Invalid molecules removed: 0
Valid molecules: 3666

Descriptor summary:


,MW,LogP,HBA,HBD
count,3666.000000,3666.000000,3666.000000,3666.00000
mean,392.158356,2.171382,4.784233,1.50982
std,89.853058,1.360660,1.677581,0.81429
min,129.167000,-3.449230,1.000000,0.00000
25%,335.304000,1.305205,4.000000,1.00000
50%,389.328000,2.207950,5.000000,1.00000
75%,446.463000,3.021605,6.000000,2.00000
max,1174.445000,12.047300,17.000000,9.00000


**Step 5.  Apply simple drug-like filtering and prepare fingerprints for diversity selection**

In [ ]:
# DPP-4 inhibitors can be polar/heterocyclic, so filtering is intentionally relaxed
druglike_df = df[
    (df["MW"].between(150, 650)) &
    (df["LogP"].between(-3, 7)) &
    (df["HBA"] <= 12) &
    (df["HBD"] <= 6)
].copy()

print("Drug-like candidate pool:", druglike_df.shape)

# Check activity-class distribution after filtering
print("\nActivity class distribution after drug-like filtering:")
display(druglike_df[class_col].value_counts())

# Morgan fingerprint generator for diversity selection
# Chirality is included to match the QSAR fingerprint setting
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=True
)

def mol_to_fp(mol):
    """
    Convert an RDKit molecule to a Morgan fingerprint.
    Returns None if fingerprint generation fails.
    """
    try:
        return morgan_generator.GetFingerprint(mol)
    except:
        return None

# Generate fingerprints
druglike_df["fp"] = druglike_df["mol"].apply(mol_to_fp)

# Remove compounds with failed fingerprints, if any
failed_fp_count = druglike_df["fp"].isna().sum()
druglike_df = druglike_df.dropna(subset=["fp"]).copy()

print("\nFailed fingerprints removed:", failed_fp_count)
print("Fingerprints prepared:", druglike_df["fp"].notna().sum())

# Display only columns that actually exist
display_cols = [
    "molecule_chembl_id",
    "pref_name",
    smiles_col,
    activity_col,
    class_col,
    "MW",
    "LogP",
    "HBA",
    "HBD"
]

display_cols = [col for col in display_cols if col in druglike_df.columns]

display(druglike_df[display_cols].head())

Drug-like candidate pool: (3625, 31)

Activity class distribution after drug-like filtering:


,count
activity_class,
High,2088
Moderate,906
Low,631



Failed fingerprints removed: 0
Fingerprints prepared: 3625


,molecule_chembl_id,pref_name,parent_smiles,pIC50_median,activity_class,MW,LogP,HBA,HBD
0,CHEMBL4246951,NaN,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,4.552067,Low,318.134,2.61660,5,1
1,CHEMBL260701,NaN,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H](C)N1,7.221849,High,276.340,-0.57342,4,1
2,CHEMBL3913618,NaN,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1ccccc1C#N,7.795880,High,417.473,0.37738,6,1
3,CHEMBL5277490,NaN,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3ccccc3n1)c(=O)n2C,8.876148,High,458.526,0.75732,7,1
4,CHEMBL1650422,NaN,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,8.301030,High,333.395,1.07208,5,1


**Export drug-like candidate pool to CSV**

In [ ]:
export_cols = [
    "molecule_chembl_id",
    "pref_name",
    smiles_col,
    activity_col,
    class_col,
    "MW",
    "LogP",
    "HBA",
    "HBD"
]

# Keep only columns that exist
export_cols = [col for col in export_cols if col in druglike_df.columns]

druglike_export_df = druglike_df[export_cols].copy()

druglike_export_df.to_csv(
    "DPP4_druglike_candidate_pool.csv",
    index=False
)

print("CSV saved: DPP4_druglike_candidate_pool.csv")
print("Exported shape:", druglike_export_df.shape)

CSV saved: DPP4_druglike_candidate_pool.csv
Exported shape: (3625, 9)


**Step 6. Define diversity selection function**

In [ ]:
def select_diverse_compounds(df_input, n=5, similarity_cutoff=0.85):
    """
    Select chemically diverse compounds using Morgan fingerprint Tanimoto similarity.

    Compounds are selected in the existing order of df_input.
    Therefore, sort df_input before using this function.

    Parameters
    ----------
    df_input : pandas DataFrame
        Input dataframe containing an 'fp' column.
    n : int
        Number of compounds to select.
    similarity_cutoff : float
        Maximum allowed Tanimoto similarity to already selected compounds.

    Returns
    -------
    pandas DataFrame
        Selected diverse compounds.
    """

    selected_rows = []
    selected_fps = []

    if df_input.empty:
        return pd.DataFrame()

    for _, row in df_input.iterrows():
        fp = row["fp"]

        # Skip compounds with missing fingerprint
        if fp is None:
            continue

        # Always select the first compound
        if len(selected_fps) == 0:
            selected_rows.append(row)
            selected_fps.append(fp)

        else:
            # Compare this compound to all already selected compounds
            max_sim = max(
                DataStructs.TanimotoSimilarity(fp, selected_fp)
                for selected_fp in selected_fps
            )

            # Keep compound only if it is not too similar
            if max_sim <= similarity_cutoff:
                selected_rows.append(row)
                selected_fps.append(fp)

        # Stop once enough compounds are selected
        if len(selected_rows) >= n:
            break

    return pd.DataFrame(selected_rows).reset_index(drop=True)

**Step 7. Select high-, moderate-, and low-activity compounds**

In [ ]:
N_KNOWN = 2
N_HIGH = 3
N_MODERATE = 2
N_LOW = 2
SIMILARITY_CUTOFF = 0.85


# 1. Known DPP-4 inhibitors
# Select known inhibitors from druglike_df so they already have descriptors and fingerprints

if "pref_name_upper" not in druglike_df.columns and "pref_name" in druglike_df.columns:
    druglike_df["pref_name_upper"] = druglike_df["pref_name"].astype(str).str.upper()

if "pref_name_upper" in druglike_df.columns:
    selected_known = druglike_df[
        druglike_df["pref_name_upper"].isin(known_dpp4_names)
    ].copy()
else:
    selected_known = druglike_df.iloc[0:0].copy()

selected_known = selected_known.sort_values(
    by=activity_col,
    ascending=False
).head(N_KNOWN)

selected_known["selection_group"] = "Known DPP-4 inhibitor"


# Get known inhibitor indices to avoid duplicate selection in other groups
known_indices = selected_known.index.tolist()


# 2. Additional high-activity candidates
# Strong DPP-4 inhibitors, excluding known inhibitors

high_pool = druglike_df[
    (druglike_df[activity_col] >= 7) &
    (~druglike_df.index.isin(known_indices))
].copy()

high_pool = high_pool.sort_values(
    by=activity_col,
    ascending=False
)

selected_high = select_diverse_compounds(
    high_pool,
    n=N_HIGH,
    similarity_cutoff=SIMILARITY_CUTOFF
)

selected_high["selection_group"] = "High activity"


# 3. Moderate-activity candidates
# Select compounds close to pIC50 6.5

moderate_pool = druglike_df[
    (druglike_df[activity_col] >= 6) &
    (druglike_df[activity_col] < 7) &
    (~druglike_df.index.isin(known_indices))
].copy()

moderate_pool["distance_to_6_5"] = abs(
    moderate_pool[activity_col] - 6.5
)

moderate_pool = moderate_pool.sort_values(
    by="distance_to_6_5",
    ascending=True
)

selected_moderate = select_diverse_compounds(
    moderate_pool,
    n=N_MODERATE,
    similarity_cutoff=SIMILARITY_CUTOFF
)

selected_moderate["selection_group"] = "Moderate activity"


# 4. Low-activity candidates
# Weak inhibitors, excluding known inhibitors

low_pool = druglike_df[
    (druglike_df[activity_col] < 6) &
    (~druglike_df.index.isin(known_indices))
].copy()

low_pool = low_pool.sort_values(
    by=activity_col,
    ascending=True
)

selected_low = select_diverse_compounds(
    low_pool,
    n=N_LOW,
    similarity_cutoff=SIMILARITY_CUTOFF
)

selected_low["selection_group"] = "Low activity"


# Summary of selected groups
print("Selected known inhibitors:", selected_known.shape)
print("Selected additional high:", selected_high.shape)
print("Selected moderate:", selected_moderate.shape)
print("Selected low:", selected_low.shape)

Selected known inhibitors: (2, 33)
Selected additional high: (3, 33)
Selected moderate: (2, 34)
Selected low: (2, 33)


**Step 8. Combine final docking candidates**

In [ ]:
selected_docking = pd.concat(
    [
        selected_known,
        selected_high,
        selected_moderate,
        selected_low
    ],
    ignore_index=False
)

# Remove duplicate parent structures
# Known inhibitors are placed first, so if duplicates exist, the known-inhibitor label is kept.
selected_docking = selected_docking.drop_duplicates(
    subset=[smiles_col],
    keep="first"
).copy()

# Sort by activity
selected_docking = selected_docking.sort_values(
    by=activity_col,
    ascending=False
).reset_index(drop=True)

# Add candidate IDs
selected_docking["candidate_id"] = [
    f"C{i+1}" for i in range(selected_docking.shape[0])
]

# Columns to show
columns_to_show = [
    "candidate_id",
    "selection_group",
    "molecule_chembl_id",
    "pref_name",
    smiles_col,
    activity_col,
    class_col,
    "MW",
    "LogP",
    "HBA",
    "HBD"
]

# Keep only columns that exist
columns_to_show = [
    col for col in columns_to_show
    if col in selected_docking.columns
]

print("Final docking candidate set:", selected_docking.shape)

display(selected_docking[columns_to_show])

Final docking candidate set: (9, 35)


,candidate_id,selection_group,molecule_chembl_id,pref_name,parent_smiles,pIC50_median,activity_class,MW,LogP,HBA,HBD
0,C1,High activity,CHEMBL5914058,NaN,CC#CCn1c(N2CCC[C@@H](N)C2)nc2c1c(=O)n(Cc1nccnc1C#N)c(=O)n2C,10.096910,High,433.476,-0.44252,8,1
1,C2,High activity,CHEMBL5269668,NaN,CC#CCN1C(N2CCC[C@@H](N)C2)=NC2C1C(=O)N(Cc1cccnc1C#N)C(=O)N2Cc1cccnc1C#N,10.045757,High,536.600,0.99706,10,1
2,C3,High activity,CHEMBL186877,NaN,COc1cc(OC)cc(-c2nc(-c3ccc(Cl)cc3Cl)c(CN(C)C)c(N(C)C)n2)c1,10.000000,High,461.393,5.26220,6,0
3,C4,Known DPP-4 inhibitor,CHEMBL1779710,EVOGLIPTIN,CC(C)(C)OC[C@@H]1C(=O)NCCN1C(=O)C[C@H](N)Cc1cc(F)c(F)cc1F,9.045757,High,401.429,1.50590,4,2
4,C5,Known DPP-4 inhibitor,CHEMBL515387,GOSOGLIPTIN,O=C([C@@H]1C[C@H](N2CCN(c3ncccn3)CC2)CN1)N1CCC(F)(F)C1,7.918166,High,366.416,0.19670,6,1
5,C6,Moderate activity,CHEMBL203032,NaN,N#C[C@@H]1CCCN1C(=O)CNCCC(=O)NCc1ccc([N+](=O)[O-])cc1,6.498941,Moderate,359.386,0.70528,6,2
6,C7,Moderate activity,CHEMBL2086587,NaN,NC(=O)[C@@H]1CCCN1C(=O)CCNC(=O)c1ccccc1,6.494850,Moderate,289.335,0.28280,3,2
7,C8,Low activity,CHEMBL459438,NaN,C[C@@H](O)[C@H](N)C(=O)N1CCCC1,4.008774,Low,172.228,-0.68310,3,2
8,C9,Low activity,CHEMBL3618137,NaN,COc1ccc(-n2ncc3c(=O)n(-c4nc(-c5ccc(Cl)cc5)cs4)c(C)nc32)cc1,4.000000,Low,449.923,4.66532,6,0


**Step 9. Save docking candidate table**

In [ ]:
export_cols = [
    "candidate_id",
    "selection_group",
    "molecule_chembl_id",
    "pref_name",
    smiles_col,
    activity_col,
    class_col,
    "MW",
    "LogP",
    "HBA",
    "HBD"
]

# Keep only columns that exist
export_cols = [
    col for col in export_cols
    if col in selected_docking.columns
]

selected_docking[export_cols].to_csv(
    "DPP4_selected_docking_candidates.csv",
    index=False
)

print("CSV saved: DPP4_selected_docking_candidates.csv")
print("Exported shape:", selected_docking[export_cols].shape)

CSV saved: DPP4_selected_docking_candidates.csv
Exported shape: (9, 11)


**Step 10. Save ligand SDF**

In [ ]:
sdf_filename = "DPP4_selected_docking_candidates.sdf"

writer = Chem.SDWriter(sdf_filename)

failed_ligands = []

# ETKDGv3 settings for 3D conformer generation
embed_params = AllChem.ETKDGv3()
embed_params.randomSeed = RANDOM_STATE

for _, row in selected_docking.iterrows():
    mol = Chem.MolFromSmiles(row[smiles_col])

    if mol is None:
        failed_ligands.append(row.get("molecule_chembl_id", "unknown"))
        continue

    # Add explicit hydrogens before 3D embedding
    mol = Chem.AddHs(mol)

    # Generate 3D conformer
    embed_status = AllChem.EmbedMolecule(
        mol,
        embed_params
    )

    if embed_status != 0:
        failed_ligands.append(row.get("molecule_chembl_id", "unknown"))
        continue

    # Geometry optimization
    try:
        if AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol)
        else:
            AllChem.UFFOptimizeMolecule(mol)
    except:
        failed_ligands.append(row.get("molecule_chembl_id", "unknown"))
        continue

    # Add ligand metadata to SDF
    mol.SetProp("_Name", str(row.get("candidate_id", row.get("molecule_chembl_id", "unknown"))))
    mol.SetProp("candidate_id", str(row.get("candidate_id", "")))
    mol.SetProp("molecule_chembl_id", str(row.get("molecule_chembl_id", "")))
    mol.SetProp("pref_name", str(row.get("pref_name", "")))
    mol.SetProp("selection_group", str(row.get("selection_group", "")))
    mol.SetProp("pIC50_median", str(row.get(activity_col, "")))
    mol.SetProp("activity_class", str(row.get(class_col, "")))
    mol.SetProp("SMILES", str(row.get(smiles_col, "")))

    writer.write(mol)

writer.close()

print(f"SDF saved: {sdf_filename}")
print("Number of failed ligand embeddings/optimizations:", len(failed_ligands))
print("Failed ligands:", failed_ligands)

SDF saved: DPP4_selected_docking_candidates.sdf
Number of failed ligand embeddings/optimizations: 0
Failed ligands: []


**Step 11. Split merged ligand SDF file into individual SDF files for docking**

In [ ]:
input_sdf = "DPP4_selected_docking_candidates.sdf"
output_dir = "ligands_split"

os.makedirs(output_dir, exist_ok=True)

supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)

summary = []

for i, mol in enumerate(supplier, start=1):
    if mol is None:
        continue

    # Use candidate_id first, then _Name, then fallback ID
    if mol.HasProp("candidate_id"):
        ligand_id = mol.GetProp("candidate_id")
    elif mol.HasProp("_Name"):
        ligand_id = mol.GetProp("_Name")
    else:
        ligand_id = f"ligand_{i}"

    # Clean ligand ID for filename safety
    safe_ligand_id = (
        ligand_id
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
    )

    output_file = os.path.join(output_dir, f"{safe_ligand_id}.sdf")

    writer = Chem.SDWriter(output_file)
    writer.write(mol)
    writer.close()

    summary.append({
        "Ligand_ID": ligand_id,
        "File": output_file,
        "candidate_id": mol.GetProp("candidate_id") if mol.HasProp("candidate_id") else "",
        "molecule_chembl_id": mol.GetProp("molecule_chembl_id") if mol.HasProp("molecule_chembl_id") else "",
        "pref_name": mol.GetProp("pref_name") if mol.HasProp("pref_name") else "",
        "selection_group": mol.GetProp("selection_group") if mol.HasProp("selection_group") else "",
        "pIC50_median": mol.GetProp("pIC50_median") if mol.HasProp("pIC50_median") else "",
        "activity_class": mol.GetProp("activity_class") if mol.HasProp("activity_class") else "",
        "SMILES": mol.GetProp("SMILES") if mol.HasProp("SMILES") else Chem.MolToSmiles(mol)
    })

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    "ligand_split_summary.csv",
    index=False
)

print(f"Done. Split {len(summary_df)} ligands.")
print("Summary saved: ligand_split_summary.csv")

display(summary_df)

Done. Split 9 ligands.
Summary saved: ligand_split_summary.csv


,Ligand_ID,File,candidate_id,molecule_chembl_id,pref_name,selection_group,pIC50_median,activity_class,SMILES
0,C1,ligands_split/C1.sdf,C1,CHEMBL5914058,nan,High activity,10.096910013008056,High,CC#CCn1c(N2CCC[C@@H](N)C2)nc2c1c(=O)n(Cc1nccnc1C#N)c(=O)n2C
1,C2,ligands_split/C2.sdf,C2,CHEMBL5269668,nan,High activity,10.045757490560677,High,CC#CCN1C(N2CCC[C@@H](N)C2)=NC2C1C(=O)N(Cc1cccnc1C#N)C(=O)N2Cc1cccnc1C#N
2,C3,ligands_split/C3.sdf,C3,CHEMBL186877,nan,High activity,10.0,High,COc1cc(OC)cc(-c2nc(-c3ccc(Cl)cc3Cl)c(CN(C)C)c(N(C)C)n2)c1
3,C4,ligands_split/C4.sdf,C4,CHEMBL1779710,EVOGLIPTIN,Known DPP-4 inhibitor,9.045757490560677,High,CC(C)(C)OC[C@@H]1C(=O)NCCN1C(=O)C[C@H](N)Cc1cc(F)c(F)cc1F
4,C5,ligands_split/C5.sdf,C5,CHEMBL515387,GOSOGLIPTIN,Known DPP-4 inhibitor,7.918165923108665,High,O=C([C@@H]1C[C@H](N2CCN(c3ncccn3)CC2)CN1)N1CCC(F)(F)C1
5,C6,ligands_split/C6.sdf,C6,CHEMBL203032,nan,Moderate activity,6.498940737782249,Moderate,N#C[C@@H]1CCCN1C(=O)CNCCC(=O)NCc1ccc([N+](=O)[O-])cc1
6,C7,ligands_split/C7.sdf,C7,CHEMBL2086587,nan,Moderate activity,6.494850021680094,Moderate,NC(=O)[C@@H]1CCCN1C(=O)CCNC(=O)c1ccccc1
7,C8,ligands_split/C8.sdf,C8,CHEMBL459438,nan,Low activity,4.008773924307505,Low,C[C@@H](O)[C@H](N)C(=O)N1CCCC1
8,C9,ligands_split/C9.sdf,C9,CHEMBL3618137,nan,Low activity,4.0,Low,COc1ccc(-n2ncc3c(=O)n(-c4nc(-c5ccc(Cl)cc5)cs4)c(C)nc32)cc1
